# 🎙️ VibeVoice Colab — T4 Quickstart (1.5B)

รัน VibeVoice TTS บน Google Colab (T4 GPU) โดยใช้ **community fork** ที่มีโค้ดครบ

> ⚠️ T4 GPU รองรับแค่โมเดล **1.5B** (RAM จำกัด) และใช้ SDPA แทน flash_attention_2 (คุณภาพอาจลดลงเล็กน้อย)
>
> สำหรับคุณภาพดีที่สุด ใช้โมเดล **7B** บน GPU ที่แรงกว่า

**วิธีใช้:** Runtime → Change runtime type → เลือก **T4 GPU** → Save

**รันทีละ cell ตามลำดับ (▶ หรือ Shift+Enter)**

## Step 1: ตรวจ GPU + ติดตั้ง Environment

In [ ]:
# ตรวจ GPU
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ ไม่พบ GPU — ไปที่ Runtime > Change runtime type > เลือก T4 GPU")

In [ ]:
# Clone community fork (มีโค้ดครบ)
!git clone --quiet --depth 1 https://github.com/vibevoice-community/VibeVoice.git /content/VibeVoice
print("✅ Cloned vibevoice-community/VibeVoice")

In [ ]:
# ติดตั้ง dependencies (ใช้ pip แทน uv เพื่อความเสถียรใน Colab)
!pip install --quiet -e /content/VibeVoice

# แก้ version conflict: transformers 4.51.3 ต้องใช้ huggingface-hub <1.0
# แต่ Colab ติดตั้ง huggingface-hub 1.x ไว้ก่อนแล้ว → บังคับ downgrade
!pip install --quiet "huggingface-hub>=0.30.0,<1.0"
print("✅ Installed dependencies + fixed huggingface-hub")

In [ ]:
# ดาวน์โหลดโมเดล 1.5B (~3-5 นาที) — ใช้ snapshot_download เพื่อความแน่นอน
# (ได้ไฟล์ครบรวม config.json ลงในโฟลเดอร์ที่กำหนด)
from huggingface_hub import snapshot_download

path = snapshot_download(
    "vibevoice/VibeVoice-1.5B",
    local_dir="/content/models/VibeVoice-1.5B"
)
print("✅ Model saved to:", path)

# ตรวจสอบว่า config.json มีครบ
import os
print("config.json exists:", os.path.exists("/content/models/VibeVoice-1.5B/config.json"))

## Step 2: สร้าง Transcript (บทสนทนา)

In [ ]:
%%writefile /content/my_transcript.txt
Speaker 1: Can I try VibeVoice with my own example?
Speaker 2: Of course! VibeVoice is open-source, built to benefit everyone - you're welcome to try it out.

## Step 3: สร้างเสียง (Generate Audio)

In [ ]:
# รัน TTS จาก transcript (ใช้ path โมเดลที่ดาวน์โหลดไว้)
!python /content/VibeVoice/demo/inference_from_file.py \
    --model_path /content/models/VibeVoice-1.5B \
    --txt_path /content/my_transcript.txt \
    --speaker_names Alice Frank

# แสดงผลเสียง
from IPython.display import Audio
Audio("/content/outputs/my_transcript_generated.wav")

## Step 4: ดาวน์โหลดเสียง

In [ ]:
from google.colab import files
files.download("/content/outputs/my_transcript_generated.wav")

## ⚠️ ข้อควรระวัง

- โมเดลรองรับ **อังกฤษ + จีน** เท่านั้น (ยังไม่มีไทย)
- เสียงที่สร้างอาจมี BGM/ดนตรีประกอบแทรกมาเอง (เป็นคุณสมบัติของโมเดล)
- ใช้เพื่อการวิจัยเท่านั้น ระวังการนำไปใช้ในทางที่ผิด (deepfake)